# PIERS RM Young All-in-One reader

Read the bundled PIERS0027 sample, inspect its normalized metadata, convert it to a pandas table, and plot its weather observations. Times in the resulting dataset are UTC.

In [ ]:
from pathlib import Path
import os
import sys

GV_HOME = Path(
    os.environ.get("GV_TOOLS_HOME", Path.home() / "Desktop" / "Work" / "GV Tools")
).expanduser().resolve()
PACKAGE = GV_HOME / "gv_tools"
SOURCE = PACKAGE / "src"
if SOURCE.is_dir() and str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
if not PACKAGE.is_dir():
    raise FileNotFoundError(f"GV Tools project not found at {PACKAGE}. Set GV_TOOLS_HOME.")

INPUT_DIR = Path(os.environ.get("GV_TOOLS_SAMPLES", GV_HOME / "Samples")).expanduser()
FILES = [INPUT_DIR / "PIERS0027_WX_20260721000005.csv"]
OUTPUT_DIR = Path(os.environ.get("GV_TOOLS_OUTPUT", GV_HOME / "Output")).expanduser()
INPUT_DIR, FILES, OUTPUT_DIR


In [ ]:
import gv_tools

if gv_tools.__version__ != "0.29.4":
    raise RuntimeError(
        f"This notebook requires GV Tools 0.29.4, but the Jupyter kernel has "
        f"{gv_tools.__version__} loaded. Restart the kernel, then Run All cells."
    )

gv_tools.io.discover_aio(FILES)


## Decode the raw AIO packets

`read_raw()` exposes the decoded pandas observations plus ingest accounting before GV normalization.

In [ ]:
raw = gv_tools.io.read_aio_raw(FILES)
print("Source files:", raw.source_files)
print("Rejected rows:", raw.rejected_rows)
print("Duplicate rows:", raw.duplicate_rows)
raw.observations


## Normalize by UTC day

In [ ]:
aio = gv_tools.io.read_aio(FILES)
aio


In [ ]:
from gv_tools import write_product

created = write_product(aio, OUTPUT_DIR, formats=('csv',), day='2026-07-21')
created

In [ ]:
observations = aio.to_dataframe()
observations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
observations[['wind_speed']].plot(ax=axes[0], legend=False, color='tab:blue')
axes[0].set_ylabel('Wind speed (m/s)')
observations[['wind_from_direction']].plot(ax=axes[1], legend=False, color='tab:orange')
axes[1].set_ylabel('Direction (deg)')
observations[['air_temperature', 'relative_humidity']].plot(ax=axes[2])
axes[2].set_ylabel('Temperature / RH')
axes[2].set_xlabel('UTC time')
fig.suptitle('PIERS0027 RM Young All-in-One sample')
fig.tight_layout()
plt.show()